In [ ]:
print("hello")

In [ ]:

import cv2
import matplotlib.pyplot as plt

image = cv2.imread(r"C:\Users\PC-LAB1\Desktop\mu\course\New folder\paper_noisy_camera_man.webp")

bright = cv2.convertScaleAbs(
    image, alpha=1.0, beta=40
)

contrast = cv2.convertScaleAbs(
    image, alpha=2, beta=0)

plt.subplot(1, 3, 1)
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.subplot(1, 3, 2)
plt.imshow(cv2.cvtColor(bright, cv2.COLOR_BGR2RGB))
plt.subplot(1, 3, 3)
plt.imshow(cv2.cvtColor(contrast, cv2.COLOR_BGR2RGB))
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

image = cv2.imread(r"C:\Users\PC-LAB1\Desktop\mu\course\New folder\dex.jpg")
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
hist, bins = np.histogram(
    gray.ravel(), bins=256, range=(0, 256)
)
plt.subplot(2, 1, 1)
plt.imshow(rgb, cmap='gray')
plt.axis('off')
plt.subplot(2, 1, 2)
plt.plot(hist)
plt.title("Grayscale Histogram")
plt.xlabel("Intensity")
plt.ylabel("Frequency")
plt.show()
cv2.imshow("Gray", gray)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Read image
image = cv2.imread(
    r"C:\Users\PC-LAB1\Desktop\mu\course\New folder\noisy_lena.webp"
)

# Convert to grayscale
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Convert to binary image
_, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

# Create kernel
kernel = np.ones((5, 5), np.uint8)

# Morphological operations
eroded = cv2.erode(binary, kernel, iterations=1)
dilated = cv2.dilate(binary, kernel, iterations=1)

opening = cv2.morphologyEx(
    binary,
    cv2.MORPH_OPEN,
    kernel
)

closing = cv2.morphologyEx(
    binary,
    cv2.MORPH_CLOSE,
    kernel
)

# Display results
plt.subplot(2, 3, 1)
plt.imshow(binary, cmap="gray")
plt.title("Original")
plt.axis("off")

plt.subplot(2, 3, 2)
plt.imshow(eroded, cmap="gray")
plt.title("Erosion")
plt.axis("off")

plt.subplot(2, 3, 3)
plt.imshow(dilated, cmap="gray")
plt.title("Dilation")
plt.axis("off")

plt.subplot(2, 3, 4)
plt.imshow(opening, cmap="gray")
plt.title("Opening")
plt.axis("off")

plt.subplot(2, 3, 5)
plt.imshow(closing, cmap="gray")
plt.title("Closing")
plt.axis("off")

plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Read image
image = cv2.imread(
    r"C:\Users\PC-LAB1\Desktop\mu\course\New folder\noisy_lena.webp"
)

# Convert to grayscale
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

### basic thresholding
_, binary = cv2.threshold(
    gray, 127, 255, cv2.THRESH_BINARY
)

cv2.imshow("output/binary.png", binary)
cv2.waitKey(0)
cv2.destroyAllWindows()

adaptive = cv2.adaptiveThreshold(
    gray, 255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    11, 2
)

cv2.imshow("output/adaptive.png", adaptive)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [ ]:
import cv2
import matplotlib.pyplot as plt

image = cv2.imread(r"C:\Users\PC-LAB1\Desktop\mu\course\New folder\imahe_with_shape.jpeg")

gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(
    gray, (5, 5), 0
)

edges = cv2.Canny(
    blurred, 50, 150
)

cv2.imshow("Gray", edges)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path


# ============================================================
# 1. CONFIGURATION
# ============================================================

IMAGE_DIR = "dataset/images"
MASK_DIR = "dataset/masks"

IMAGE_SIZE = (512, 512)

# Minimum connected-component area
MIN_AREA = 50

# Morphological kernel
KERNEL_SIZE = 3

# Supported image extensions
IMAGE_EXTENSIONS = [
    "*.jpg",
    "*.jpeg",
    "*.png",
    "*.bmp",
    "*.tif",
    "*.tiff"
]


# ============================================================
# 2. LOAD IMAGE FILES
# ============================================================

def get_image_files(folder):

    files = []

    for extension in IMAGE_EXTENSIONS:
        files.extend(Path(folder).glob(extension))

    return sorted(files)


image_files = get_image_files(IMAGE_DIR)

print("Number of images:", len(image_files))


# ============================================================
# 3. FIND CORRESPONDING MASK
# ============================================================

def find_mask(image_path):

    image_name = image_path.stem

    for extension in [
        ".png",
        ".jpg",
        ".jpeg",
        ".bmp",
        ".tif",
        ".tiff"
    ]:

        mask_path = Path(MASK_DIR) / (image_name + extension)

        if mask_path.exists():
            return mask_path

    return None


# ============================================================
# 4. LOAD AND PREPROCESS IMAGE
# ============================================================

def preprocess_image(image_path):

    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(
            f"Could not read image: {image_path}"
        )

    # Resize
    image = cv2.resize(
        image,
        IMAGE_SIZE,
        interpolation=cv2.INTER_AREA
    )

    # Normalize
    image = cv2.normalize(
        image,
        None,
        0,
        255,
        cv2.NORM_MINMAX
    )

    # Grayscale
    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    return image, gray


# ============================================================
# 5. LOAD GROUND-TRUTH MASK
# ============================================================

def load_mask(mask_path):

    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE
    )

    if mask is None:
        raise ValueError(
            f"Could not read mask: {mask_path}"
        )

    mask = cv2.resize(
        mask,
        IMAGE_SIZE,
        interpolation=cv2.INTER_NEAREST
    )

    # Convert mask to binary
    mask = np.where(
        mask > 127,
        255,
        0
    ).astype(np.uint8)

    return mask


# ============================================================
# 6. NOISE REDUCTION
# ============================================================

def noise_reduction(gray, method="gaussian"):

    if method == "gaussian":

        return cv2.GaussianBlur(
            gray,
            (5, 5),
            0
        )

    elif method == "median":

        return cv2.medianBlur(
            gray,
            5
        )

    elif method == "bilateral":

        return cv2.bilateralFilter(
            gray,
            9,
            75,
            75
        )

    else:
        raise ValueError(
            "Unknown noise reduction method"
        )


# ============================================================
# 7. CONTRAST ENHANCEMENT
# ============================================================

def enhance_contrast(image, method="clahe"):

    if method == "histogram":

        return cv2.equalizeHist(image)

    elif method == "clahe":

        clahe = cv2.createCLAHE(
            clipLimit=2.0,
            tileGridSize=(8, 8)
        )

        return clahe.apply(image)

    else:
        raise ValueError(
            "Unknown contrast enhancement method"
        )


# ============================================================
# 8. CRACK ENHANCEMENT
# ============================================================

def crack_enhancement(image, method="canny"):

    if method == "sobel":

        sobel_x = cv2.Sobel(
            image,
            cv2.CV_64F,
            1,
            0,
            ksize=3
        )

        sobel_y = cv2.Sobel(
            image,
            cv2.CV_64F,
            0,
            1,
            ksize=3
        )

        magnitude = cv2.magnitude(
            sobel_x.astype(np.float32),
            sobel_y.astype(np.float32)
        )

        return cv2.convertScaleAbs(magnitude)


    elif method == "scharr":

        scharr_x = cv2.Scharr(
            image,
            cv2.CV_64F,
            1,
            0
        )

        scharr_y = cv2.Scharr(
            image,
            cv2.CV_64F,
            0,
            1
        )

        magnitude = cv2.magnitude(
            scharr_x.astype(np.float32),
            scharr_y.astype(np.float32)
        )

        return cv2.convertScaleAbs(magnitude)


    elif method == "canny":

        return cv2.Canny(
            image,
            50,
            150
        )


    elif method == "laplacian":

        result = cv2.Laplacian(
            image,
            cv2.CV_64F
        )

        return cv2.convertScaleAbs(result)


    else:

        raise ValueError(
            "Unknown crack enhancement method"
        )


# ============================================================
# 9. THRESHOLDING
# ============================================================

def threshold_image(image, method="otsu"):

    if method == "global":

        _, binary = cv2.threshold(
            image,
            100,
            255,
            cv2.THRESH_BINARY
        )

        return binary


    elif method == "otsu":

        _, binary = cv2.threshold(
            image,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

        return binary


    elif method == "adaptive":

        return cv2.adaptiveThreshold(
            image,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            11,
            2
        )


    else:

        raise ValueError(
            "Unknown thresholding method"
        )


# ============================================================
# 10. MORPHOLOGICAL PROCESSING
# ============================================================

def morphological_processing(
    binary,
    operation="closing"
):

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (KERNEL_SIZE, KERNEL_SIZE)
    )

    if operation == "opening":

        result = cv2.morphologyEx(
            binary,
            cv2.MORPH_OPEN,
            kernel
        )


    elif operation == "closing":

        result = cv2.morphologyEx(
            binary,
            cv2.MORPH_CLOSE,
            kernel,
            iterations=2
        )


    elif operation == "erosion":

        result = cv2.erode(
            binary,
            kernel,
            iterations=1
        )


    elif operation == "dilation":

        result = cv2.dilate(
            binary,
            kernel,
            iterations=1
        )


    else:

        raise ValueError(
            "Unknown morphological operation"
        )

    return result


# ============================================================
# 11. CONNECTED COMPONENT ANALYSIS
# ============================================================

def remove_small_components(
    binary,
    min_area=50
):

    num_labels, labels, stats, centroids = \
        cv2.connectedComponentsWithStats(
            binary,
            connectivity=8
        )

    output = np.zeros_like(binary)

    for label in range(1, num_labels):

        area = stats[
            label,
            cv2.CC_STAT_AREA
        ]

        if area >= min_area:

            output[
                labels == label
            ] = 255

    return output


# ============================================================
# 12. CONTOUR EXTRACTION
# ============================================================

def extract_contours(binary):

    contours, hierarchy = cv2.findContours(
        binary,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    return contours


# ============================================================
# 13. FINAL CRACK SEGMENTATION
# ============================================================

def final_segmentation(binary):

    contours = extract_contours(binary)

    output = np.zeros_like(binary)

    for contour in contours:

        area = cv2.contourArea(contour)

        if area >= MIN_AREA:

            cv2.drawContours(
                output,
                [contour],
                -1,
                255,
                thickness=cv2.FILLED
            )

    return output


# ============================================================
# 14. EVALUATION METRICS
# ============================================================

def calculate_metrics(
    ground_truth,
    prediction
):

    gt = ground_truth > 0
    pred = prediction > 0

    TP = np.logical_and(
        gt,
        pred
    ).sum()

    TN = np.logical_and(
        ~gt,
        ~pred
    ).sum()

    FP = np.logical_and(
        ~gt,
        pred
    ).sum()

    FN = np.logical_and(
        gt,
        ~pred
    ).sum()


    # IoU
    union = TP + FP + FN

    if union == 0:
        iou = 1.0
    else:
        iou = TP / union


    # Dice / F1
    denominator = (
        2 * TP + FP + FN
    )

    if denominator == 0:
        dice = 1.0
    else:
        dice = (
            2 * TP
        ) / denominator


    # Precision
    if TP + FP == 0:
        precision = 0.0
    else:
        precision = TP / (TP + FP)


    # Recall
    if TP + FN == 0:
        recall = 0.0
    else:
        recall = TP / (TP + FN)


    # Accuracy
    total = TP + TN + FP + FN

    accuracy = (
        TP + TN
    ) / total


    return {
        "IoU": iou,
        "Dice": dice,
        "Precision": precision,
        "Recall": recall,
        "Accuracy": accuracy
    }


# ============================================================
# 15. COMPLETE PIPELINE
# ============================================================

def crack_pipeline(
    image_path,
    noise_method="gaussian",
    contrast_method="clahe",
    enhancement_method="canny",
    threshold_method="otsu",
    morphology_method="closing"
):

    image, gray = preprocess_image(
        image_path
    )

    # Noise reduction
    filtered = noise_reduction(
        gray,
        noise_method
    )

    # Contrast enhancement
    enhanced = enhance_contrast(
        filtered,
        contrast_method
    )

    # Crack enhancement
    cracks = crack_enhancement(
        enhanced,
        enhancement_method
    )

    # Threshold
    binary = threshold_image(
        cracks,
        threshold_method
    )

    # Morphology
    morphological = morphological_processing(
        binary,
        morphology_method
    )

    # Connected components
    cleaned = remove_small_components(
        morphological,
        MIN_AREA
    )

    # Final segmentation
    final_mask = final_segmentation(
        cleaned
    )

    return {
        "original": image,
        "gray": gray,
        "filtered": filtered,
        "enhanced": enhanced,
        "cracks": cracks,
        "binary": binary,
        "morphological": morphological,
        "cleaned": cleaned,
        "final": final_mask
    }


# ============================================================
# 16. RUN PIPELINE ON ALL IMAGES
# ============================================================

results = []

for image_path in image_files:

    mask_path = find_mask(image_path)

    if mask_path is None:

        print(
            f"WARNING: No mask found for {image_path.name}"
        )

        continue


    try:

        ground_truth = load_mask(
            mask_path
        )


        output = crack_pipeline(
            image_path,

            noise_method="gaussian",

            contrast_method="clahe",

            enhancement_method="canny",

            threshold_method="otsu",

            morphology_method="closing"
        )


        metrics = calculate_metrics(
            ground_truth,
            output["final"]
        )


        metrics["image"] = image_path.name

        results.append(metrics)


        print(
            f"{image_path.name}: "
            f"IoU={metrics['IoU']:.4f}, "
            f"Dice={metrics['Dice']:.4f}, "
            f"Precision={metrics['Precision']:.4f}, "
            f"Recall={metrics['Recall']:.4f}"
        )


    except Exception as e:

        print(
            f"ERROR processing {image_path.name}: {e}"
        )


# ============================================================
# 17. RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(results)

if len(results_df) > 0:

    print("\n==============================")
    print("DATASET RESULTS")
    print("==============================")

    print(
        results_df.to_string(
            index=False
        )
    )


    print("\n==============================")
    print("AVERAGE PERFORMANCE")
    print("==============================")

    metric_columns = [
        "IoU",
        "Dice",
        "Precision",
        "Recall",
        "Accuracy"
    ]

    print(
        results_df[
            metric_columns
        ].mean()
    )


    # Save results
    results_df.to_csv(
        "crack_segmentation_results.csv",
        index=False
    )


# ============================================================
# 18. VISUALIZE ONE IMAGE
# ============================================================

if len(image_files) > 0:

    example_image = image_files[0]

    example_mask_path = find_mask(
        example_image
    )

    if example_mask_path is not None:

        gt = load_mask(
            example_mask_path
        )

        output = crack_pipeline(
            example_image,
            noise_method="gaussian",
            contrast_method="clahe",
            enhancement_method="canny",
            threshold_method="otsu",
            morphology_method="closing"
        )


        fig, axes = plt.subplots(
            3,
            4,
            figsize=(16, 12)
        )


        visualization = [
            ("Original", output["original"]),
            ("Grayscale", output["gray"]),
            ("Filtered", output["filtered"]),
            ("CLAHE", output["enhanced"]),
            ("Crack Enhancement", output["cracks"]),
            ("Threshold", output["binary"]),
            ("Morphology", output["morphological"]),
            ("Cleaned", output["cleaned"]),
            ("Ground Truth", gt),
            ("Final Segmentation", output["final"])
        ]


        for ax, (title, img) in zip(
            axes.ravel(),
            visualization
        ):

            if len(img.shape) == 2:

                ax.imshow(
                    img,
                    cmap="gray"
                )

            else:

                # OpenCV BGR → RGB
                ax.imshow(
                    cv2.cvtColor(
                        img,
                        cv2.COLOR_BGR2RGB
                    )
                )

            ax.set_title(title)
            ax.axis("off")


        # Hide unused axes
        for ax in axes.ravel()[
            len(visualization):
        ]:

            ax.axis("off")


        plt.tight_layout()
        plt.show()

In [ ]:
import cv2
import glob

image_folder = r"C:\Users\PC-LAB1\Desktop\mu\course\New folder\day3miniproject\data\images"
mask_folder = r"C:\Users\PC-LAB1\Desktop\mu\course\New folder\day3miniproject\data\masks"
images = [cv2.imread(path) for path in glob.glob(image_folder + r"\*") ]
masks = [cv2.imread(path, 0) for path in glob.glob(mask_folder + r"\*") ]
binary_images = []


for image in images:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    binary = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        11,
        2
    )

    binary_images.append(binary)

  

In [ ]:
import cv2
import glob
import numpy as np

image_folder = r"\\192.168.1.10\shear all\day3\train\images\images"
mask_folder = r"\\192.168.1.10\shear all\day3\train\masks\masks"

image_paths = glob.glob(image_folder + r"\*")
mask_paths = glob.glob(mask_folder + r"\*")

#images = [cv2.imread(path) for path in image_paths]
#masks = [cv2.imread(path, 0) for path in mask_paths]

image = cv2.imread(str(image_path))

mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE
    )


binary_images = []
accuracies = []
precisions = []
recalls = []
f1_scores = []
ious = []
dice_scores = []

# Morphological kernel
kernel = np.ones((3, 3), np.uint8)

for image, mask in zip(images, masks):

    # -------------------------
    # 1. Grayscale
    # -------------------------
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # -------------------------
    # 2. Filtering
    # -------------------------
    gray = cv2.medianBlur(gray, 5)

    # -------------------------
    # 3. Adaptive threshold
    # -------------------------
    binary = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        11,
        2
    )

    # -------------------------
    # 4. Morphological opening
    # Removes small noise
    # -------------------------
    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        kernel
    )

    # -------------------------
    # 5. Morphological closing
    # Fills small gaps
    # -------------------------
    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_CLOSE,
        kernel
    )

    binary_images.append(binary)

    # -------------------------
    # 6. Make sure mask is binary
    # -------------------------
    mask_binary = (mask > 127).astype(np.uint8)

    prediction = (binary > 127).astype(np.uint8)

    # -------------------------
    # 7. Confusion matrix
    # -------------------------
    TP = np.sum((prediction == 1) & (mask_binary == 1))
    TN = np.sum((prediction == 0) & (mask_binary == 0))
    FP = np.sum((prediction == 1) & (mask_binary == 0))
    FN = np.sum((prediction == 0) & (mask_binary == 1))

    # -------------------------
    # 8. Metrics
    # -------------------------
    accuracy = (TP + TN) / (TP + TN + FP + FN)

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0

    recall = TP / (TP + FN) if (TP + FN) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    iou = (
        TP / (TP + FP + FN)
        if (TP + FP + FN) > 0
        else 0
    )

    dice = (
        2 * TP / (2 * TP + FP + FN)
        if (2 * TP + FP + FN) > 0
        else 0
    )

    accuracies.append(accuracy)
    precisions.append(precision)
    recalls.append(recall)
    f1_scores.append(f1)
    ious.append(iou)
    dice_scores.append(dice)


# =====================================
# Average results over all images
# =====================================

print("Average Accuracy :", np.mean(accuracies))
print("Average Precision:", np.mean(precisions))
print("Average Recall   :", np.mean(recalls))
print("Average F1 Score :", np.mean(f1_scores))
print("Average IoU      :", np.mean(ious))
print("Average Dice     :", np.mean(dice_scores))

In [ ]:

index=7000
cv2.imshow("Image", images[index])
cv2.imshow("Mask", masks[index])
cv2.imshow("Binary", binary_images[index])
cv2.waitKey(0)
cv2.destroyAllWindows() 

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np


# ============================================================
# SETTINGS
# ============================================================

image_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\images\images"
)

mask_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\masks\masks"
)


NUMBER_OF_IMAGES = 5

# Simple threshold
THRESHOLD = 99

# Median filter AFTER threshold
MEDIAN_KERNEL = 3

# Erosion
EROSION_KERNEL = 3
EROSION_ITERATIONS = 1


# ============================================================
# GET IMAGE FILES
# ============================================================

valid_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

image_paths = sorted([
    path
    for path in image_folder.glob("*")
    if path.suffix.lower() in valid_extensions
])

print("Total images found:", len(image_paths))


# ============================================================
# PROCESS IMAGES
# ============================================================

for image_path in image_paths[:NUMBER_OF_IMAGES]:

    print("\n" + "=" * 70)
    print("IMAGE:", image_path.name)
    print("=" * 70)

    # ========================================================
    # FIND MASK
    # ========================================================

    mask_path = mask_folder / image_path.name

    if not mask_path.exists():
        print("Mask not found:", image_path.name)
        continue

    # ========================================================
    # READ IMAGE AND MASK
    # ========================================================

    image = cv2.imread(str(image_path))

    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None or mask is None:
        print("Could not read:", image_path.name)
        continue

    # ========================================================
    # 1. GRAYSCALE
    # ========================================================

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    # ========================================================
    # 2. SIMPLE THRESHOLD
    # ========================================================

    _, binary = cv2.threshold(
        gray,
        THRESHOLD,
        255,
        cv2.THRESH_BINARY_INV
    )

    # ========================================================
    # 3. MEDIAN FILTER
    # ========================================================

    median_binary = cv2.medianBlur(
        binary,
        MEDIAN_KERNEL
    )

    # ========================================================
    # 4. EROSION
    # ========================================================

    erosion_kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (EROSION_KERNEL, EROSION_KERNEL)
    )

    eroded = cv2.erode(
        median_binary,
        erosion_kernel,
        iterations=EROSION_ITERATIONS
    )

    # ========================================================
    # 5. GROUND TRUTH MASK
    # ========================================================

    _, mask_binary = cv2.threshold(
        mask,
        127,
        255,
        cv2.THRESH_BINARY
    )

    # ========================================================
    # 6. CALCULATE IoU
    # ========================================================

    TP = np.logical_and(
        mask_binary == 255,
        eroded == 255
    ).sum()

    FP = np.logical_and(
        mask_binary == 0,
        eroded == 255
    ).sum()

    FN = np.logical_and(
        mask_binary == 255,
        eroded == 0
    ).sum()

    denominator = TP + FP + FN

    IoU = (
        TP / denominator
        if denominator > 0
        else 0
    )

    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print("TP:", TP)
    print("FP:", FP)
    print("FN:", FN)

    print(
        f"IoU: {IoU:.4f} "
        f"({IoU * 100:.2f}%)"
    )

    # ========================================================
    # DISPLAY ALL STAGES
    # ========================================================

    plt.figure(figsize=(20, 5))

    # --------------------------------------------------------
    # 1. ORIGINAL
    # --------------------------------------------------------

    plt.subplot(1, 6, 1)

    plt.imshow(
        cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )
    )

    plt.title(
        f"1. Original\n{image_path.name}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # 2. GRAYSCALE
    # --------------------------------------------------------

    plt.subplot(1, 6, 2)

    plt.imshow(
        gray,
        cmap="gray"
    )

    plt.title("2. Grayscale")

    plt.axis("off")

    # --------------------------------------------------------
    # 3. SIMPLE THRESHOLD
    # --------------------------------------------------------

    plt.subplot(1, 6, 3)

    plt.imshow(
        binary,
        cmap="gray"
    )

    plt.title(
        f"3. Simple Threshold\n"
        f"Threshold={THRESHOLD}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # 4. MEDIAN FILTER
    # --------------------------------------------------------

    plt.subplot(1, 6, 4)

    plt.imshow(
        median_binary,
        cmap="gray"
    )

    plt.title(
        f"4. Median Filter\n"
        f"Kernel={MEDIAN_KERNEL}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # 5. EROSION
    # --------------------------------------------------------

    plt.subplot(1, 6, 5)

    plt.imshow(
        eroded,
        cmap="gray"
    )

    plt.title(
        f"5. Erosion\n"
        f"Kernel={EROSION_KERNEL}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # 6. GROUND TRUTH
    # --------------------------------------------------------

    plt.subplot(1, 6, 6)

    plt.imshow(
        mask_binary,
        cmap="gray"
    )

    plt.title(
        f"6. Ground Truth\n"
        f"IoU={IoU:.2%}"
    )

    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path


# ============================================================
# 1. CONFIGURATION
# ============================================================

IMAGE_DIR = r"\\192.168.1.10\shear all\day3\train\images\images"
MASK_DIR = r"\\192.168.1.10\shear all\day3\train\masks\masks"

IMAGE_SIZE = (512, 512)

# Minimum connected-component area
MIN_AREA = 50

# Morphological kernel
KERNEL_SIZE = 3

# Supported image extensions
IMAGE_EXTENSIONS = [
    "*.jpg",
    "*.jpeg",
    "*.png",
    "*.bmp",
    "*.tif",
    "*.tiff"
]


# ============================================================
# 2. LOAD IMAGE FILES
# ============================================================

def get_image_files(folder):

    files = []

    for extension in IMAGE_EXTENSIONS:
        files.extend(Path(folder).glob(extension))

    return sorted(files)


image_files = get_image_files(IMAGE_DIR)

print("Number of images:", len(image_files))


# ============================================================
# 3. FIND CORRESPONDING MASK
# ============================================================

def find_mask(image_path):

    image_name = image_path.stem

    for extension in [
        ".png",
        ".jpg",
        ".jpeg",
        ".bmp",
        ".tif",
        ".tiff"
    ]:

        mask_path = Path(MASK_DIR) / (image_name + extension)

        if mask_path.exists():
            return mask_path

    return None


# ============================================================
# 4. LOAD AND PREPROCESS IMAGE
# ============================================================

def preprocess_image(image_path):

    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(
            f"Could not read image: {image_path}"
        )

    # Resize
    image = cv2.resize(
        image,
        IMAGE_SIZE,
        interpolation=cv2.INTER_AREA
    )

    # Normalize
    image = cv2.normalize(
        image,
        None,
        0,
        255,
        cv2.NORM_MINMAX
    )

    # Grayscale
    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    return image, gray


# ============================================================
# 5. LOAD GROUND-TRUTH MASK
# ============================================================

def load_mask(mask_path):

    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE
    )

    if mask is None:
        raise ValueError(
            f"Could not read mask: {mask_path}"
        )

    mask = cv2.resize(
        mask,
        IMAGE_SIZE,
        interpolation=cv2.INTER_NEAREST
    )

    # Convert mask to binary
    mask = np.where(
        mask > 127,
        255,
        0
    ).astype(np.uint8)

    return mask


# ============================================================
# 6. NOISE REDUCTION
# ============================================================

def noise_reduction(gray, method="gaussian"):

    if method == "gaussian":

        return cv2.GaussianBlur(
            gray,
            (5, 5),
            0
        )

    elif method == "median":

        return cv2.medianBlur(
            gray,
            5
        )

    elif method == "bilateral":

        return cv2.bilateralFilter(
            gray,
            9,
            75,
            75
        )

    else:
        raise ValueError(
            "Unknown noise reduction method"
        )


# ============================================================
# 7. CONTRAST ENHANCEMENT
# ============================================================

def enhance_contrast(image, method="clahe"):

    if method == "histogram":

        return cv2.equalizeHist(image)

    elif method == "clahe":

        clahe = cv2.createCLAHE(
            clipLimit=2.0,
            tileGridSize=(8, 8)
        )

        return clahe.apply(image)

    else:
        raise ValueError(
            "Unknown contrast enhancement method"
        )


# ============================================================
# 8. CRACK ENHANCEMENT
# ============================================================

def crack_enhancement(image, method="canny"):

    if method == "sobel":

        sobel_x = cv2.Sobel(
            image,
            cv2.CV_64F,
            1,
            0,
            ksize=3
        )

        sobel_y = cv2.Sobel(
            image,
            cv2.CV_64F,
            0,
            1,
            ksize=3
        )

        magnitude = cv2.magnitude(
            sobel_x.astype(np.float32),
            sobel_y.astype(np.float32)
        )

        return cv2.convertScaleAbs(magnitude)


    elif method == "scharr":

        scharr_x = cv2.Scharr(
            image,
            cv2.CV_64F,
            1,
            0
        )

        scharr_y = cv2.Scharr(
            image,
            cv2.CV_64F,
            0,
            1
        )

        magnitude = cv2.magnitude(
            scharr_x.astype(np.float32),
            scharr_y.astype(np.float32)
        )

        return cv2.convertScaleAbs(magnitude)


    elif method == "canny":

        return cv2.Canny(
            image,
            50,
            150
        )


    elif method == "laplacian":

        result = cv2.Laplacian(
            image,
            cv2.CV_64F
        )

        return cv2.convertScaleAbs(result)


    else:

        raise ValueError(
            "Unknown crack enhancement method"
        )


# ============================================================
# 9. THRESHOLDING
# ============================================================

def threshold_image(image, method="otsu"):

    if method == "global":

        _, binary = cv2.threshold(
            image,
            100,
            255,
            cv2.THRESH_BINARY
        )

        return binary


    elif method == "otsu":

        _, binary = cv2.threshold(
            image,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

        return binary


    elif method == "adaptive":

        return cv2.adaptiveThreshold(
            image,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            11,
            2
        )


    else:

        raise ValueError(
            "Unknown thresholding method"
        )


# ============================================================
# 10. MORPHOLOGICAL PROCESSING
# ============================================================

def morphological_processing(
    binary,
    operation="closing"
):

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (KERNEL_SIZE, KERNEL_SIZE)
    )

    if operation == "opening":

        result = cv2.morphologyEx(
            binary,
            cv2.MORPH_OPEN,
            kernel
        )


    elif operation == "closing":

        result = cv2.morphologyEx(
            binary,
            cv2.MORPH_CLOSE,
            kernel,
            iterations=2
        )


    elif operation == "erosion":

        result = cv2.erode(
            binary,
            kernel,
            iterations=1
        )


    elif operation == "dilation":

        result = cv2.dilate(
            binary,
            kernel,
            iterations=1
        )


    else:

        raise ValueError(
            "Unknown morphological operation"
        )

    return result


# ============================================================
# 11. CONNECTED COMPONENT ANALYSIS
# ============================================================

def remove_small_components(
    binary,
    min_area=50
):

    num_labels, labels, stats, centroids = \
        cv2.connectedComponentsWithStats(
            binary,
            connectivity=8
        )

    output = np.zeros_like(binary)

    for label in range(1, num_labels):

        area = stats[
            label,
            cv2.CC_STAT_AREA
        ]

        if area >= min_area:

            output[
                labels == label
            ] = 255

    return output


# ============================================================
# 12. CONTOUR EXTRACTION
# ============================================================

def extract_contours(binary):

    contours, hierarchy = cv2.findContours(
        binary,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    return contours


# ============================================================
# 13. FINAL CRACK SEGMENTATION
# ============================================================

def final_segmentation(binary):

    contours = extract_contours(binary)

    output = np.zeros_like(binary)

    for contour in contours:

        area = cv2.contourArea(contour)

        if area >= MIN_AREA:

            cv2.drawContours(
                output,
                [contour],
                -1,
                255,
                thickness=cv2.FILLED
            )

    return output


# ============================================================
# 14. EVALUATION METRICS
# ============================================================

def calculate_metrics(
    ground_truth,
    prediction
):

    gt = ground_truth > 0
    pred = prediction > 0

    TP = np.logical_and(
        gt,
        pred
    ).sum()

    TN = np.logical_and(
        ~gt,
        ~pred
    ).sum()

    FP = np.logical_and(
        ~gt,
        pred
    ).sum()

    FN = np.logical_and(
        gt,
        ~pred
    ).sum()


    # IoU
    union = TP + FP + FN

    if union == 0:
        iou = 1.0
    else:
        iou = TP / union


    # Dice / F1
    denominator = (
        2 * TP + FP + FN
    )

    if denominator == 0:
        dice = 1.0
    else:
        dice = (
            2 * TP
        ) / denominator


    # Precision
    if TP + FP == 0:
        precision = 0.0
    else:
        precision = TP / (TP + FP)


    # Recall
    if TP + FN == 0:
        recall = 0.0
    else:
        recall = TP / (TP + FN)


    # Accuracy
    total = TP + TN + FP + FN

    accuracy = (
        TP + TN
    ) / total


    return {
        "IoU": iou,
        "Dice": dice,
        "Precision": precision,
        "Recall": recall,
        "Accuracy": accuracy
    }


# ============================================================
# 15. COMPLETE PIPELINE
# ============================================================

def crack_pipeline(
    image_path,
    noise_method="gaussian",
    contrast_method="clahe",
    enhancement_method="canny",
    threshold_method="otsu",
    morphology_method="closing"
):

    image, gray = preprocess_image(
        image_path
    )

    # Noise reduction
    filtered = noise_reduction(
        gray,
        noise_method
    )

    # Contrast enhancement
    enhanced = enhance_contrast(
        filtered,
        contrast_method
    )

    # Crack enhancement
    cracks = crack_enhancement(
        enhanced,
        enhancement_method
    )

    # Threshold
    binary = threshold_image(
        cracks,
        threshold_method
    )

    # Morphology
    morphological = morphological_processing(
        binary,
        morphology_method
    )

    # Connected components
    cleaned = remove_small_components(
        morphological,
        MIN_AREA
    )

    # Final segmentation
    final_mask = final_segmentation(
        cleaned
    )

    return {
        "original": image,
        "gray": gray,
        "filtered": filtered,
        "enhanced": enhanced,
        "cracks": cracks,
        "binary": binary,
        "morphological": morphological,
        "cleaned": cleaned,
        "final": final_mask
    }


# ============================================================
# 16. RUN PIPELINE ON ALL IMAGES
# ============================================================

results = []

for image_path in image_files:

    mask_path = find_mask(image_path)

    if mask_path is None:

        print(
            f"WARNING: No mask found for {image_path.name}"
        )

        continue


    try:

        ground_truth = load_mask(
            mask_path
        )


        output = crack_pipeline(
            image_path,

            noise_method="gaussian",

            contrast_method="clahe",

            enhancement_method="canny",

            threshold_method="otsu",

            morphology_method="closing"
        )


        metrics = calculate_metrics(
            ground_truth,
            output["final"]
        )


        metrics["image"] = image_path.name

        results.append(metrics)


        print(
            f"{image_path.name}: "
            f"IoU={metrics['IoU']:.4f}, "
            f"Dice={metrics['Dice']:.4f}, "
            f"Precision={metrics['Precision']:.4f}, "
            f"Recall={metrics['Recall']:.4f}"
        )


    except Exception as e:

        print(
            f"ERROR processing {image_path.name}: {e}"
        )


# ============================================================
# 17. RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(results)

if len(results_df) > 0:

    print("\n==============================")
    print("DATASET RESULTS")
    print("==============================")

    print(
        results_df.to_string(
            index=False
        )
    )


    print("\n==============================")
    print("AVERAGE PERFORMANCE")
    print("==============================")

    metric_columns = [
        "IoU",
        "Dice",
        "Precision",
        "Recall",
        "Accuracy"
    ]

    print(
        results_df[
            metric_columns
        ].mean()
    )


    # Save results
    results_df.to_csv(
        "crack_segmentation_results.csv",
        index=False
    )


# ============================================================
# 18. VISUALIZE ONLY 3 IMAGES
# ============================================================

if len(image_files) > 0:

    example_image = image_files[0]

    example_mask_path = find_mask(
        example_image
    )

    if example_mask_path is not None:

        gt = load_mask(
            example_mask_path
        )

        output = crack_pipeline(
            example_image,
            noise_method="gaussian",
            contrast_method="clahe",
            enhancement_method="canny",
            threshold_method="otsu",
            morphology_method="closing"
        )

        # عرض 3 صور فقط:
        # 1. Original Image
        # 2. Ground Truth Mask
        # 3. Binary Image المعدلة بعد المعالجة
        visualization = [
            ("Original Image", output["original"]),
            ("Ground Truth Mask", gt),
            ("Modified Binary Image", output["cleaned"])
        ]

        fig, axes = plt.subplots(
            1,
            3,
            figsize=(15, 5)
        )

        for ax, (title, img) in zip(
            axes.ravel(),
            visualization
        ):

            if len(img.shape) == 2:

                ax.imshow(
                    img,
                    cmap="gray"
                )

            else:

                # OpenCV BGR → RGB
                ax.imshow(
                    cv2.cvtColor(
                        img,
                        cv2.COLOR_BGR2RGB
                    )
                )

            ax.set_title(title)
            ax.axis("off")

        plt.tight_layout()
        plt.show()


In [ ]:

import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np


# ============================================================
# SETTINGS
# ============================================================

image_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\images\images"
)

mask_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\masks\masks"
)


# ============================================================
# IMAGE RANGE
# ============================================================

# يبدأ العد من 0
START_INDEX = 6900

# غير هذا الرقم حسب عدد الصور التي تريدها
END_INDEX = 7000


# ============================================================
# PROCESSING SETTINGS
# ============================================================

# Simple threshold
THRESHOLD = 99

# Median filter AFTER threshold
MEDIAN_KERNEL = 3

# Erosion
EROSION_KERNEL = 3
EROSION_ITERATIONS = 1


# ============================================================
# GET IMAGE FILES
# ============================================================

valid_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

image_paths = sorted([
    path
    for path in image_folder.glob("*")
    if path.suffix.lower() in valid_extensions
])

print("Total images found:", len(image_paths))

print(
    f"Processing images from index "
    f"{START_INDEX} to {END_INDEX - 1}"
)


# ============================================================
# CHECK RANGE
# ============================================================

if START_INDEX < 0:
    START_INDEX = 0

if END_INDEX > len(image_paths):
    END_INDEX = len(image_paths)

if START_INDEX >= END_INDEX:
    raise ValueError(
        "Invalid image range. "
        "Make sure START_INDEX < END_INDEX."
    )


# ============================================================
# PROCESS IMAGES
# ============================================================

iou_values = []

selected_images = image_paths[START_INDEX:END_INDEX]

for image_path in selected_images:

    print("\n" + "=" * 70)
    print("IMAGE:", image_path.name)
    print("=" * 70)

    # ========================================================
    # FIND MASK
    # ========================================================

    mask_path = mask_folder / image_path.name

    if not mask_path.exists():
        print("Mask not found:", image_path.name)
        continue

    # ========================================================
    # READ IMAGE AND MASK
    # ========================================================

    image = cv2.imread(str(image_path))

    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None or mask is None:
        print("Could not read:", image_path.name)
        continue

    # ========================================================
    # CHECK IMAGE SIZE
    # ========================================================

    if image.shape[:2] != mask.shape[:2]:
        print("Size mismatch:")
        print("Image:", image.shape[:2])
        print("Mask :", mask.shape[:2])
        continue

    # ========================================================
    # 1. GRAYSCALE
    # ========================================================

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    # ========================================================
    # 2. SIMPLE THRESHOLD
    # ========================================================

    _, binary = cv2.threshold(
        gray,
        THRESHOLD,
        255,
        cv2.THRESH_BINARY_INV
    )

    # ========================================================
    # 3. MEDIAN FILTER
    # ========================================================

    median_binary = cv2.medianBlur(
        binary,
        MEDIAN_KERNEL
    )

    # ========================================================
    # 4. EROSION
    # ========================================================

    erosion_kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (EROSION_KERNEL, EROSION_KERNEL)
    )

    eroded = cv2.erode(
        median_binary,
        erosion_kernel,
        iterations=EROSION_ITERATIONS
    )

    # ========================================================
    # 5. GROUND TRUTH MASK
    # ========================================================

    _, mask_binary = cv2.threshold(
        mask,
        127,
        255,
        cv2.THRESH_BINARY
    )

    # ========================================================
    # 6. CALCULATE IoU
    # ========================================================

    TP = np.logical_and(
        mask_binary == 255,
        eroded == 255
    ).sum()

    FP = np.logical_and(
        mask_binary == 0,
        eroded == 255
    ).sum()

    FN = np.logical_and(
        mask_binary == 255,
        eroded == 0
    ).sum()

    denominator = TP + FP + FN

    IoU = (
        TP / denominator
        if denominator > 0
        else 0
    )

    iou_values.append(IoU)

    print(f"TP  = {TP}")
    print(f"FP  = {FP}")
    print(f"FN  = {FN}")
    print(f"IoU = {IoU:.4f}")
    print(f"IoU = {IoU * 100:.2f}%")


    # ========================================================
    # DISPLAY ONLY 3 IMAGES
    # ========================================================

    plt.figure(figsize=(15, 5))

    # --------------------------------------------------------
    # 1. ORIGINAL IMAGE
    # --------------------------------------------------------

    plt.subplot(1, 3, 1)

    plt.imshow(
        cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    )

    plt.title("Original Image")
    plt.axis("off")


    # --------------------------------------------------------
    # 2. GROUND TRUTH MASK
    # --------------------------------------------------------

    plt.subplot(1, 3, 2)

    plt.imshow(
        mask_binary,
        cmap="gray"
    )

    plt.title("Ground Truth Mask")
    plt.axis("off")


    # --------------------------------------------------------
    # 3. MODIFIED BINARY IMAGE
    # --------------------------------------------------------

    plt.subplot(1, 3, 3)

    plt.imshow(
        eroded,
        cmap="gray"
    )

    plt.title(
        f"Modified Binary\nIoU = {IoU:.4f}"
    )

    plt.axis("off")


    plt.suptitle(
        f"{image_path.name}"
    )

    plt.tight_layout()
    plt.show()


# ============================================================
# FINAL SUMMARY
# ============================================================

if iou_values:

    mean_iou = np.mean(iou_values)

    print("\n" + "=" * 70)
    print("FINAL RESULTS")
    print("=" * 70)

    print(
        f"Images processed: {len(iou_values)}"
    )

    print(
        f"Mean IoU: {mean_iou:.4f}"
    )

    print(
        f"Mean IoU: {mean_iou * 100:.2f}%"
    )

else:

    print("\nNo valid images were processed.")



In [ ]:

import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np


# ============================================================
# SETTINGS
# ============================================================

image_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\images\images"
)

mask_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\masks\masks"
)


# ============================================================
# IMAGE RANGE
# ============================================================

# يبدأ العد من 0
START_INDEX = 300

# غير هذا الرقم حسب الصور التي تريد معالجتها
END_INDEX = 401


# ============================================================
# NOISE REDUCTION
# ============================================================

# الخيارات:
# "gaussian"
# "median"
# "bilateral"

NOISE_REDUCTION_METHOD = "gaussian"


# ============================================================
# THRESHOLD
# ============================================================

THRESHOLD = 99


# ============================================================
# MEDIAN FILTER AFTER THRESHOLD
# ============================================================

MEDIAN_KERNEL = 3


# ============================================================
# EROSION
# ============================================================

EROSION_KERNEL = 3
EROSION_ITERATIONS = 1


# ============================================================
# NOISE REDUCTION FUNCTION
# ============================================================

def noise_reduction(gray, method="gaussian"):

    if method == "gaussian":

        return cv2.GaussianBlur(
            gray,
            (5, 5),
            0
        )

    elif method == "median":

        return cv2.medianBlur(
            gray,
            5
        )

    elif method == "bilateral":

        return cv2.bilateralFilter(
            gray,
            9,
            75,
            75
        )

    else:
        raise ValueError(
            "Unknown noise reduction method"
        )


# ============================================================
# GET IMAGE FILES
# ============================================================

valid_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

image_paths = sorted([
    path
    for path in image_folder.glob("*")
    if path.suffix.lower() in valid_extensions
])

print("Total images found:", len(image_paths))


# ============================================================
# CHECK IMAGE RANGE
# ============================================================

if START_INDEX < 0:
    START_INDEX = 0

if END_INDEX > len(image_paths):
    END_INDEX = len(image_paths)

if START_INDEX >= END_INDEX:
    raise ValueError(
        "Invalid image range. "
        "Make sure START_INDEX < END_INDEX."
    )


selected_images = image_paths[
    START_INDEX:END_INDEX
]

print(
    f"Images selected: "
    f"{START_INDEX} → {END_INDEX - 1}"
)

print(
    f"Number of images to process: "
    f"{len(selected_images)}"
)

print(
    f"Noise reduction method: "
    f"{NOISE_REDUCTION_METHOD}"
)


# ============================================================
# STORE IoU VALUES
# ============================================================

iou_values = []


# ============================================================
# PROCESS IMAGES
# ============================================================

for image_number, image_path in enumerate(
    selected_images,
    start=START_INDEX
):

    print("\n" + "=" * 70)
    print(
        f"IMAGE INDEX: {image_number}"
    )
    print(
        f"IMAGE: {image_path.name}"
    )
    print("=" * 70)


    # ========================================================
    # FIND MASK
    # ========================================================

    mask_path = mask_folder / image_path.name

    if not mask_path.exists():

        print(
            "Mask not found:",
            image_path.name
        )

        continue


    # ========================================================
    # READ IMAGE
    # ========================================================

    image = cv2.imread(
        str(image_path)
    )

    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE
    )


    if image is None or mask is None:

        print(
            "Could not read:",
            image_path.name
        )

        continue


    # ========================================================
    # CHECK SIZE
    # ========================================================

    if image.shape[:2] != mask.shape[:2]:

        print("Size mismatch")

        print(
            "Image:",
            image.shape[:2]
        )

        print(
            "Mask:",
            mask.shape[:2]
        )

        continue


    # ========================================================
    # 1. GRAYSCALE
    # ========================================================

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )


    # ========================================================
    # 2. NOISE REDUCTION
    # ========================================================

    filtered_gray = noise_reduction(
        gray,
        method=NOISE_REDUCTION_METHOD
    )


    # ========================================================
    # 3. SIMPLE THRESHOLD
    # ========================================================

    _, binary = cv2.threshold(
        filtered_gray,
        THRESHOLD,
        255,
        cv2.THRESH_BINARY_INV
    )


    # ========================================================
    # 4. MEDIAN FILTER
    # ========================================================

    median_binary = cv2.medianBlur(
        binary,
        MEDIAN_KERNEL
    )


    # ========================================================
    # 5. EROSION
    # ========================================================

    erosion_kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (
            EROSION_KERNEL,
            EROSION_KERNEL
        )
    )

    eroded = cv2.erode(
        median_binary,
        erosion_kernel,
        iterations=EROSION_ITERATIONS
    )


    # ========================================================
    # 6. GROUND TRUTH MASK
    # ========================================================

    _, mask_binary = cv2.threshold(
        mask,
        127,
        255,
        cv2.THRESH_BINARY
    )


    # ========================================================
    # 7. CALCULATE IoU
    # ========================================================

    TP = np.logical_and(
        mask_binary == 255,
        eroded == 255
    ).sum()

    FP = np.logical_and(
        mask_binary == 0,
        eroded == 255
    ).sum()

    FN = np.logical_and(
        mask_binary == 255,
        eroded == 0
    ).sum()


    denominator = TP + FP + FN


    if denominator > 0:

        IoU = TP / denominator

    else:

        IoU = 0


    # إضافة IoU إلى القائمة
    iou_values.append(IoU)


    # ========================================================
    # PRINT IoU
    # ========================================================

    print(
        f"TP  = {TP}"
    )

    print(
        f"FP  = {FP}"
    )

    print(
        f"FN  = {FN}"
    )

    print(
        f"IoU = {IoU:.4f}"
    )

    print(
        f"IoU = {IoU * 100:.2f}%"
    )


    # ========================================================
    # DISPLAY ONLY 3 IMAGES
    # ========================================================

    plt.figure(figsize=(15, 5))


    # --------------------------------------------------------
    # ORIGINAL
    # --------------------------------------------------------

    plt.subplot(1, 3, 1)

    plt.imshow(
        cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )
    )

    plt.title(
        "Original Image"
    )

    plt.axis("off")


    # --------------------------------------------------------
    # GROUND TRUTH MASK
    # --------------------------------------------------------

    plt.subplot(1, 3, 2)

    plt.imshow(
        mask_binary,
        cmap="gray"
    )

    plt.title(
        "Ground Truth Mask"
    )

    plt.axis("off")


    # --------------------------------------------------------
    # FINAL BINARY
    # --------------------------------------------------------

    plt.subplot(1, 3, 3)

    plt.imshow(
        eroded,
        cmap="gray"
    )

    plt.title(
        f"Modified Binary\nIoU = {IoU:.4f}"
    )

    plt.axis("off")


    plt.suptitle(
        image_path.name
    )

    plt.tight_layout()

    plt.show()


# ============================================================
# AVERAGE IoU
# ============================================================

if len(iou_values) > 0:

    average_iou = np.mean(
        iou_values
    )

    print("\n")
    print("=" * 70)
    print("FINAL RESULTS")
    print("=" * 70)

    print(
        f"Images processed: "
        f"{len(iou_values)}"
    )

    print(
        f"Average IoU: "
        f"{average_iou:.4f}"
    )

    print(
        f"Average IoU: "
        f"{average_iou * 100:.2f}%"
    )

    print("=" * 70)

else:

    print(
        "\nNo valid images were processed."
    )


In [ ]:

import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np


# =========================================================
# SETTINGS
# =========================================================

IMAGE_FOLDER = Path(r"\\192.168.1.10\shear all\day3\train\images\images")
MASK_FOLDER = Path(r"\\192.168.1.10\shear all\day3\train\masks\masks")

# Range of images
START_INDEX = 0
END_INDEX = 100

# Threshold
THRESHOLD = 99

# Noise reduction method:
# "gaussian", "median", or "bilateral"
NOISE_REDUCTION_METHOD = "gaussian"

# Minimum area for keeping detected objects
# Increase this value to remove more small noise
MIN_AREA = 50

# Median filter after threshold
MEDIAN_KERNEL = 3

# Erosion kernel
EROSION_KERNEL_SIZE = 3
EROSION_ITERATIONS = 1


# =========================================================
# NOISE REDUCTION FUNCTION
# =========================================================

def noise_reduction(gray, method="gaussian"):

    if method == "gaussian":
        return cv2.GaussianBlur(
            gray,
            (5, 5),
            0
        )

    elif method == "median":
        return cv2.medianBlur(
            gray,
            5
        )

    elif method == "bilateral":
        return cv2.bilateralFilter(
            gray,
            9,
            75,
            75
        )

    else:
        raise ValueError(
            "Unknown noise reduction method. "
            "Use: gaussian, median, or bilateral."
        )


# =========================================================
# REMOVE SMALL OBJECTS
# =========================================================

def remove_small_objects(binary, min_area=50):

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary,
        connectivity=8
    )

    cleaned = np.zeros_like(binary)

    for i in range(1, num_labels):

        area = stats[i, cv2.CC_STAT_AREA]

        if area >= min_area:
            cleaned[labels == i] = 255

    return cleaned


# =========================================================
# GET IMAGE FILES
# =========================================================

extensions = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff"]

image_paths = []

for ext in extensions:
    image_paths.extend(IMAGE_FOLDER.glob(ext))

image_paths = sorted(image_paths)

selected_paths = image_paths[START_INDEX:END_INDEX]

print(f"Total images found: {len(image_paths)}")
print(
    f"Processing images from index "
    f"{START_INDEX} to {END_INDEX - 1}"
)
print(f"Selected images: {len(selected_paths)}")
print()


# =========================================================
# IOU RESULTS
# =========================================================

iou_values = []


# =========================================================
# PROCESS IMAGES
# =========================================================

for image_path in selected_paths:

    # -----------------------------------------------------
    # Read image
    # -----------------------------------------------------

    image = cv2.imread(str(image_path))

    if image is None:
        print(f"Could not read image: {image_path.name}")
        continue

    # -----------------------------------------------------
    # Find corresponding mask
    # -----------------------------------------------------

    mask_path = MASK_FOLDER / image_path.name

    if not mask_path.exists():

        print(
            f"Mask not found for: {image_path.name}"
        )

        continue

    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE
    )

    if mask is None:

        print(
            f"Could not read mask: {mask_path.name}"
        )

        continue

    # -----------------------------------------------------
    # Check image / mask size
    # -----------------------------------------------------

    if image.shape[:2] != mask.shape[:2]:

        print(
            f"Size mismatch: {image_path.name}"
        )

        continue

    # =====================================================
    # ORIGINAL IMAGE -> GRAYSCALE
    # =====================================================

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    # =====================================================
    # NOISE REDUCTION
    # =====================================================

    filtered_gray = noise_reduction(
        gray,
        method=NOISE_REDUCTION_METHOD
    )

    # =====================================================
    # THRESHOLD
    # =====================================================

    _, binary = cv2.threshold(
        filtered_gray,
        THRESHOLD,
        255,
        cv2.THRESH_BINARY_INV
    )

    # =====================================================
    # MEDIAN FILTER AFTER THRESHOLD
    # =====================================================

    binary = cv2.medianBlur(
        binary,
        MEDIAN_KERNEL
    )

    # =====================================================
    # REMOVE SMALL OBJECTS / NOISE
    # =====================================================

    binary = remove_small_objects(
        binary,
        min_area=MIN_AREA
    )

    # =====================================================
    # EROSION
    # =====================================================

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (
            EROSION_KERNEL_SIZE,
            EROSION_KERNEL_SIZE
        )
    )

    eroded = cv2.erode(
        binary,
        kernel,
        iterations=EROSION_ITERATIONS
    )

    # =====================================================
    # GROUND TRUTH MASK
    # =====================================================

    _, mask_binary = cv2.threshold(
        mask,
        127,
        255,
        cv2.THRESH_BINARY
    )

    # =====================================================
    # IOU
    # =====================================================

    TP = np.logical_and(
        mask_binary == 255,
        eroded == 255
    ).sum()

    FP = np.logical_and(
        mask_binary == 0,
        eroded == 255
    ).sum()

    FN = np.logical_and(
        mask_binary == 255,
        eroded == 0
    ).sum()

    denominator = TP + FP + FN

    if denominator > 0:
        iou = TP / denominator
    else:
        iou = 0

    iou_values.append(iou)

    # =====================================================
    # DISPLAY ONLY 3 IMAGES
    # =====================================================

    plt.figure(figsize=(15, 5))

    # Original
    plt.subplot(1, 3, 1)
    plt.imshow(
        cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    )
    plt.title("Original Image")
    plt.axis("off")

    # Ground truth mask
    plt.subplot(1, 3, 2)
    plt.imshow(
        mask_binary,
        cmap="gray"
    )
    plt.title("Ground Truth Mask")
    plt.axis("off")

    # Final modified binary
    plt.subplot(1, 3, 3)
    plt.imshow(
        eroded,
        cmap="gray"
    )
    plt.title(
        f"Modified Binary\nIoU = {iou:.3f}"
    )
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    print(
        f"{image_path.name} -> "
        f"IoU = {iou:.4f}"
    )


# =========================================================
# AVERAGE IOU
# =========================================================

if len(iou_values) > 0:

    average_iou = np.mean(iou_values)

    print("\n" + "=" * 50)
    print("RESULTS")
    print("=" * 50)

    print(
        f"Number of processed images: "
        f"{len(iou_values)}"
    )

    print(
        f"Average IoU: "
        f"{average_iou:.4f}"
    )

    print(
        f"Average IoU (%): "
        f"{average_iou * 100:.2f}%"
    )

    print("=" * 50)

else:

    print(
        "\nNo valid images were processed."
    )
